# Notebook 8 – Feature Scaling

This notebook covers scaling numeric features in `customer_transactions_raw.csv`, comparing techniques, and showing how the outliers identified in Notebook 6 (e.g. the 999,999,999 income value) distort some scalers far more than others.

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['purchase_amount_clean'] = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(df['purchase_amount_clean'], errors='coerce')
df[['age_numeric', 'annual_income', 'purchase_amount_clean', 'quantity']].describe()

,age_numeric,annual_income,purchase_amount_clean,quantity
count,947.00000,9.430000e+02,1000.000000,980.000000
mean,38.37698,3.251967e+06,238.573880,4.995918
std,14.55290,5.633967e+07,1641.123073,2.607834
min,-5.00000,-5.000000e+04,-100.000000,-1.000000
25%,30.00000,5.238569e+04,62.805000,3.000000
50%,38.00000,6.592116e+04,104.040000,5.000000
75%,45.00000,7.913065e+04,171.362500,7.000000
max,200.00000,1.000000e+09,25000.000000,9.000000


## 1. Why Feature Scaling?

Feature scaling puts numeric columns onto a comparable range or distribution. Without it, columns with naturally larger numeric ranges (like `annual_income`, which reaches into the millions) can dominate columns with small ranges (like `quantity`, which only spans roughly 1–9), even if both are equally important to the underlying pattern the model should learn.

In [11]:
df[['age_numeric', 'annual_income', 'purchase_amount_clean', 'quantity']].agg(['min', 'max', 'mean', 'std'])

,age_numeric,annual_income,purchase_amount_clean,quantity
min,-5.00000,-5.000000e+04,-100.000000,-1.000000
max,200.00000,1.000000e+09,25000.000000,9.000000
mean,38.37698,3.251967e+06,238.573880,4.995918
std,14.55290,5.633967e+07,1641.123073,2.607834


## 2. Scale-Sensitive Algorithms

These algorithms compute distances or gradients directly from feature magnitudes, so unscaled features with large ranges dominate the result:

- **K-Nearest Neighbors (KNN):** distance calculations are dominated by whichever feature has the largest numeric range.
- **K-Means clustering:** cluster assignment is based on Euclidean distance, same issue as KNN.
- **Support Vector Machines (SVM):** the margin-maximizing objective is sensitive to feature scale.
- **Linear/Logistic Regression with regularization (Ridge, Lasso):** the penalty term treats all coefficients equally, so unscaled features get penalized unfairly relative to their actual importance.
- **Neural networks / gradient descent-based models:** unscaled inputs can cause unstable or slow convergence.
- **PCA:** finds directions of maximum variance, which will be dominated by high-magnitude columns if not scaled.

## 3. Scale-Insensitive Algorithms

These algorithms make decisions based on relative ordering or splits rather than absolute magnitude, so they don't require scaling:

- **Decision Trees**
- **Random Forests**
- **Gradient Boosted Trees (XGBoost, LightGBM, CatBoost)**
- **Naive Bayes** (works on probability distributions per feature independently)

Scaling these models is harmless but unnecessary — it won't change split points or predictions in tree-based models.

## 4. Normalization

Normalization typically refers to **Min-Max scaling**, which rescales values into a fixed range (usually [0, 1]) based on the minimum and maximum of the data:

$$x' = \\frac{x - x_{min}}{x_{max} - x_{min}}$$

It preserves the shape of the original distribution but is **very sensitive to outliers**, since a single extreme min or max stretches the whole scale.

## 5. Standardization

Standardization (Z-score scaling) centers data around a mean of 0 with a standard deviation of 1:

$$x' = \\frac{x - \\mu}{\\sigma}$$

It does not bound values to a fixed range, and unlike Min-Max scaling it doesn't require knowing the min/max in advance, but it is still affected by outliers because both the mean and standard deviation are pulled by extreme values.

## 6. Min-Max Scaling

Min-Max scaling (see Section 4) is best suited for data that is roughly uniformly distributed within a known, meaningful range, and where the algorithm needs bounded input (e.g. neural network inputs that expect [0, 1]). It should be avoided when the data has extreme outliers.

In [12]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())
df['quantity_minmax'] = min_max_scale(df['quantity'].clip(lower=0))
df[['quantity', 'quantity_minmax']].describe()

,quantity,quantity_minmax
count,980.000000,980.000000
mean,4.995918,0.555669
std,2.607834,0.288560
min,-1.000000,0.000000
25%,3.000000,0.333333
50%,5.000000,0.555556
75%,7.000000,0.777778
max,9.000000,1.000000


## 7. StandardScaler

scikit-learn's `StandardScaler` implements standardization (Section 5): it subtracts the mean and divides by the standard deviation, using the full dataset's mean and std, including any outliers present.

**When to use:** roughly normally distributed data with no extreme outliers, or as the default choice for linear models, logistic regression, and PCA when outliers have already been handled.

In [13]:
from sklearn.preprocessing import StandardScaler
scaler_standard = StandardScaler()
age_valid = df.loc[(df['age_numeric'] >= 0) & (df['age_numeric'] <= 100), ['age_numeric']]
age_standard_scaled = scaler_standard.fit_transform(age_valid)
pd.DataFrame(age_standard_scaled, columns=['age_standard_scaled']).describe()

,age_standard_scaled
count,9.400000e+02
mean,5.291276e-17
std,1.000532e+00
min,-3.394977e+00
25%,-7.100718e-01
50%,5.902984e-03
75%,6.323809e-01
max,3.406783e+00


## 8. MinMaxScaler

scikit-learn's `MinMaxScaler` implements Min-Max scaling (Section 4/6). By default it maps values to [0, 1], but the target range is configurable.

**When to use:** bounded-input requirements (e.g. image pixel-style inputs, neural network layers expecting [0, 1]), and only after outliers have been addressed, since a single extreme value compresses every other value into a tiny sub-range.

In [14]:
from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler()
quantity_valid = df.loc[df['quantity'] >= 0, ['quantity']]
quantity_minmax_scaled = scaler_minmax.fit_transform(quantity_valid)
pd.DataFrame(quantity_minmax_scaled, columns=['quantity_minmax_scaled']).describe()

,quantity_minmax_scaled
count,975.000000
mean,0.503333
std,0.322350
min,0.000000
25%,0.250000
50%,0.500000
75%,0.750000
max,1.000000


## 9. RobustScaler

scikit-learn's `RobustScaler` centers data using the **median** and scales using the **IQR (interquartile range)** instead of the mean and standard deviation:

$$x' = \\frac{x - \\text{median}}{\\text{IQR}}$$

Because the median and IQR are not pulled by extreme values the way the mean and standard deviation are, this scaler is **much less sensitive to outliers**.

**When to use:** any column with known outliers or heavy skew, such as `annual_income` or `purchase_amount_clean` in this dataset, especially when you want to retain the outlier rows rather than remove them first.

In [15]:
from sklearn.preprocessing import RobustScaler
scaler_robust = RobustScaler()
income_all = df[['annual_income']].dropna()
income_robust_scaled = scaler_robust.fit_transform(income_all)
pd.DataFrame(income_robust_scaled, columns=['income_robust_scaled']).describe()

,income_robust_scaled
count,943.000000
mean,119.126981
std,2106.553080
min,-4.334319
25%,-0.506094
50%,0.000000
75%,0.493906
max,37387.764453


## 10. MaxAbsScaler

scikit-learn's `MaxAbsScaler` divides each value by the maximum absolute value in the column, mapping data into [-1, 1] without shifting/centering it. It preserves sparsity (zeros stay zero), which makes it a good fit for sparse data.

**When to use:** data that is already centered around zero or contains meaningful negative values, and sparse matrices (e.g. one-hot encoded or TF-IDF features) where preserving zero entries matters. Like Min-Max scaling, it is sensitive to outliers because the scale is set entirely by the single largest-magnitude value.

In [16]:
from sklearn.preprocessing import MaxAbsScaler
scaler_maxabs = MaxAbsScaler()
purchase_valid = df[['purchase_amount_clean']].dropna()
purchase_maxabs_scaled = scaler_maxabs.fit_transform(purchase_valid)
pd.DataFrame(purchase_maxabs_scaled, columns=['purchase_maxabs_scaled']).describe()

,purchase_maxabs_scaled
count,1000.000000
mean,0.009543
std,0.065645
min,-0.004000
25%,0.002512
50%,0.004162
75%,0.006855
max,1.000000


## 11. How Outliers Affect Scaling

The `annual_income` column contains a genuine data error identified in Notebook 6: a repeated sentinel value of 999,999,999. The comparison below shows what happens to `StandardScaler` and `MinMaxScaler` when that outlier is included versus excluded, compared against the outlier-resistant `RobustScaler`.

In [18]:
income_with_outlier = df[['annual_income']].dropna()
income_without_outlier = df.loc[df['annual_income'] < 900000000, ['annual_income']].dropna()
standard_with = StandardScaler().fit_transform(income_with_outlier)
standard_without = StandardScaler().fit_transform(income_without_outlier)
minmax_with = MinMaxScaler().fit_transform(income_with_outlier)
minmax_without = MinMaxScaler().fit_transform(income_without_outlier)
robust_with = RobustScaler().fit_transform(income_with_outlier)
robust_without = RobustScaler().fit_transform(income_without_outlier)
comparison = pd.DataFrame({
    'scaler': ['StandardScaler', 'StandardScaler', 'MinMaxScaler', 'MinMaxScaler', 'RobustScaler', 'RobustScaler'],
    'outlier_included': [True, False, True, False, True, False],
    'scaled_min': [standard_with.min(), standard_without.min(), minmax_with.min(), minmax_without.min(), robust_with.min(), robust_without.min()],
    'scaled_max': [standard_with.max(), standard_without.max(), minmax_with.max(), minmax_without.max(), robust_with.max(), robust_without.max()],
    'scaled_std_of_bulk_95pct': [
        np.percentile(standard_with, 95) - np.percentile(standard_with, 5),
        np.percentile(standard_without, 95) - np.percentile(standard_without, 5),
        np.percentile(minmax_with, 95) - np.percentile(minmax_with, 5),
        np.percentile(minmax_without, 95) - np.percentile(minmax_without, 5),
        np.percentile(robust_with, 95) - np.percentile(robust_with, 5),
        np.percentile(robust_without, 95) - np.percentile(robust_without, 5),
    ],
})
comparison

,scaler,outlier_included,scaled_min,scaled_max,scaled_std_of_bulk_95pct
0,StandardScaler,True,-0.058639,17.701151,0.001222
1,StandardScaler,False,-0.745137,30.390548,0.422094
2,MinMaxScaler,True,0.000000,1.000000,0.000069
3,MinMaxScaler,False,0.000000,1.000000,0.013557
4,RobustScaler,True,-4.334319,37387.764453,2.573550
5,RobustScaler,False,-4.348582,185.179557,2.569356


**What this shows:**

- **StandardScaler** — including the outlier inflates the mean and standard deviation, which compresses almost every normal-income customer into a tiny band near zero while the outlier sits many standard deviations away. The 95th–5th percentile spread of the *bulk* of the data shrinks dramatically once the outlier is included, meaning normal variation between customers becomes nearly invisible to the model.
- **MinMaxScaler** — this is affected the most. Since the scaler stretches everything between the observed min and max, one value of 999,999,999 forces every realistic income (tens of thousands) down near 0.0000x, destroying almost all resolution among genuine customers.
- **RobustScaler** — barely changes between the two runs, since the median and IQR are computed from the middle 50% of the data and are not pulled by a single extreme value.

**Conclusion:** when outliers cannot be fully removed or are only partially addressed, `RobustScaler` is the safer default. `StandardScaler` and `MinMaxScaler` should only be trusted after outliers have been properly investigated and treated (as in Notebook 6), since both derive their scale directly from statistics (mean/std, min/max) that a single extreme value can distort.

## 12. Comparing the Scaling Techniques

In [19]:
scaler_comparison = pd.DataFrame({
    'scaler': ['StandardScaler', 'MinMaxScaler', 'RobustScaler', 'MaxAbsScaler'],
    'centers_on': ['mean', 'min', 'median', 'zero (no shift)'],
    'scales_by': ['standard deviation', 'range (max - min)', 'IQR', 'max absolute value'],
    'output_range': ['unbounded', '[0, 1] by default', 'unbounded', '[-1, 1]'],
    'outlier_sensitivity': ['high', 'very high', 'low', 'high'],
    'best_for': [
        'roughly normal data, linear models, PCA',
        'bounded-input needs, uniformly distributed data',
        'data with known outliers or heavy skew',
        'sparse data, data already centered at zero',
    ],
})
scaler_comparison

,scaler,centers_on,scales_by,output_range,outlier_sensitivity,best_for
0,StandardScaler,mean,standard deviation,unbounded,high,"roughly normal data, linear models, PCA"
1,MinMaxScaler,min,range (max - min),"[0, 1] by default",very high,"bounded-input needs, uniformly distributed data"
2,RobustScaler,median,IQR,unbounded,low,data with known outliers or heavy skew
3,MaxAbsScaler,zero (no shift),max absolute value,"[-1, 1]",high,"sparse data, data already centered at zero"


In [10]:
df.to_csv('customer_transactions_scaled.csv', index=False)
df.shape

(1000, 15)